# 55장 · create_agent의 도구 등록과 메시지 왕복

create_agent는 모델 호출과 도구 실행을 연결합니다. 대화 메시지에서 도구 요청과 결과를 찾아 어느 함수가 실행됐는지 확인합니다.

운영 시간과 대출 기간을 함께 물은 뒤 AIMessage의 tool_calls와 ToolMessage를 ID로 짝지으세요. 호출 순서와 ID는 실행마다 달라질 수 있습니다.

실행을 확인한 커널은 Python 3.12.15입니다. 예제에 사용한 외부 패키지는 langchain 1.4.6, langchain-core 1.6.12, langsmith 0.14.11입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.


## 54장의 결과 열기

같은 폴더에 내려받은 노트북을 여세요. 먼저 my-course/agent-54.json를 읽습니다. 파일이 없으면 시작 셀의 완성 예시로 연습할 수 있습니다.

In [ ]:
from pathlib import Path
import json
folder = Path.cwd()
if (folder.name == "daily" or (folder.name.startswith("chapter") and folder.name[7:].isdigit())):
    folder = folder.parent
work = folder / "my-course"
work.mkdir(exist_ok=True)
source_file = work / "agent-54.json"
if source_file.exists():
    previous = json.loads(source_file.read_text(encoding="utf-8"))
else:
    previous = {'record': {'owner': 'alice', 'id': 'loan-01', 'text': '대출 기간은 14일입니다.'}}
print("내 파일" if source_file.exists() else "완성 예시", previous["record"]["text"])

완성 예시 대출 기간은 14일입니다.


## 도구 요청과 결과 메시지 연결

도구 요청에는 함수 이름과 인자, 결과를 받을 때 사용할 호출 ID가 들어 있습니다. 시간을 알아내는 일은 등록된 함수가 맡습니다. 이 예제에서는 미리 지정한 도구 요청을 Python 함수에 전달합니다.


### 셀 01 · 도구 함수의 인자와 결과

요청에 시간표 값이 있는지 먼저 보세요. 조회 뒤 어느 값이 새로 생길까요?


In [1]:
hours = {"weekday": "09:00~18:00", "weekend": "10:00~16:00"}
request = {"name": "get_hours", "args": {"day": "weekend"}, "id": "c1"}
def get_hours(day):
    return {"ok": True, "hours": hours[day]}

result = get_hours(**request["args"])
print(result["hours"])

10:00~16:00


10:00~16:00은 함수가 시간표에서 꺼낸 값입니다. 요청의 c1은 시간이 들어갈 칸이 아니라 도구 결과를 원래 요청과 짝지을 식별자입니다.


### 과제 02 · 요청과 결과의 ID 연결

두 번째 요청 c2의 응답에서 대출 기간을 꺼내세요. 목록 위치 대신 호출 ID를 쓰세요.


In [2]:
replies = {'c1': {'hours': '10:00~16:00'}, 'c2': {'loan_days': 14}}
# c2의 대출 기간을 출력하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>replies = {&#x27;c1&#x27;: {&#x27;hours&#x27;: &#x27;10:00~16:00&#x27;}, &#x27;c2&#x27;: {&#x27;loan_days&#x27;: 14}}
print(replies[&#x27;c2&#x27;][&#x27;loan_days&#x27;])</code></pre>
<p>실행 결과</p><pre><code>14</code></pre>
<p>14는 대출 규정 도구가 반환한 일수입니다. 반환 순서가 달라져도 c2로 찾아낼 수 있습니다.</p>
</details>


## 대화 기록에서 도구 호출 수 집계

먼저 제공된 메시지 유형 목록에서 ai와 tool을 따로 셉니다. 뒤의 셀 06에서는 두 함수를 create_agent에 등록해 실제 ToolMessage의 이름·ID·반환값을 읽습니다.


### 셀 03 · 도구 메시지 수 집계

운영 시간과 대출 기간을 함께 물은 뒤 AIMessage의 tool_calls와 ToolMessage를 ID로 짝지으세요. 호출 순서와 ID는 실행마다 달라질 수 있습니다.

In [3]:
roles = ["human", "ai", "tool", "ai", "tool", "ai"]
print(roles.count("tool"), roles.count("ai"))

2 3


첫 값 2는 도구 결과 메시지 수이고 둘째 값 3은 AI 메시지 수입니다. 이 목록에서는 각 AI 요청이 도구 하나만 부릅니다. 한 AI 메시지가 여러 도구를 요청할 수도 있으므로 메시지 수를 항상 호출 수로 간주할 수는 없습니다.


### 역할별 메시지 세기

<p>roles에서 ai는 세 번, tool은 두 번 나타납니다. 마지막 ai는 두 도구의 결과를 받은 뒤 답변만 쓰기 때문에 ToolMessage가 하나 더 생기지 않습니다.</p>


### 과제 04 · 대응 결과가 없는 호출

c1과 c2를 요청했지만 c1만 회신했습니다. 회신을 기다리는 ID를 출력하세요.


In [4]:
requested = ['c1', 'c2']
returned = {'c1'}
# 아직 결과가 없는 ID를 찾으세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>requested = [&#x27;c1&#x27;, &#x27;c2&#x27;]
returned = {&#x27;c1&#x27;}
print([call_id for call_id in requested if call_id not in returned])</code></pre>
<p>실행 결과</p><pre><code>[&#x27;c2&#x27;]</code></pre>
<p>[&#x27;c2&#x27;]가 남습니다. 이 상태에서 대출 기간을 확정하면 도구가 주지 않은 답을 쓰게 됩니다.</p>
</details>


## create_agent에 두 도구 등록

운영 시간과 대출 기간을 함께 물은 뒤 AIMessage의 tool_calls와 ToolMessage를 ID로 짝지으세요. 호출 순서와 ID는 실행마다 달라질 수 있습니다.

### 셀 06에서 두 도구의 반환값은 어디로 갈까요?

**주말 시간표를 요청합니다.**　첫 모델 메시지에는 도구 이름과 요청 ID가 있습니다.

```
c1 · get_hours
```

**함수가 시간표를 돌려줍니다.**　도구 메시지의 tool_call_id는 첫 요청의 ID인 c1입니다.

```
c1 · 10:00~16:00
```

**대출 기간을 요청합니다.**　다음 모델 메시지가 두 번째 요청을 냅니다.

```
c2 · get_loan_rule
```

**함수가 대출 기간을 돌려줍니다.**　이 값은 c2가 요청한 대출 일수입니다.

```
c2 · 14
```

**마지막 모델 메시지가 답을 냅니다.**　추가 도구 요청이 없어 왕복이 끝납니다.

```
모델 메시지 3개 · 도구 메시지 2개
```


### 셀 05 · 실제 모델 응답 확인

도구 요청을 담은 메시지를 만들 도우미입니다. 다음 셀에서 쓰므로 출력은 없습니다.


In [ ]:
from langchain.agents import create_agent
from langsmith import tracing_context
import os
from langchain_openai import ChatOpenAI

# 수업 커널에 %pip install langchain-openai 를 실행하고 OPENAI_API_KEY를 설정하세요.
model = ChatOpenAI(model=os.getenv("OPENAI_MODEL") or "gpt-4o-mini", timeout=45, max_retries=1)

운영 시간과 대출 기간을 함께 물은 뒤 AIMessage의 tool_calls와 ToolMessage를 ID로 짝지으세요. 호출 순서와 ID는 실행마다 달라질 수 있습니다.

### 모델과 도구 사이에서 누가 함수를 실행할까?

<p>create_agent는 모델의 도구 요청을 등록된 함수로 보내고 반환값을 다음 모델 호출에 넘깁니다. 이번에는 마지막 AIMessage의 tool_calls가 비는 시점까지 따라가세요.</p>


### 같은 도구 요청을 다시 관찰하기

<p>운영 시간과 대출 기간을 함께 물은 뒤 AIMessage의 tool_calls와 ToolMessage를 ID로 짝지으세요. 호출 순서와 ID는 실행마다 달라질 수 있습니다.</p>


### 같은 모델로 돌아가는 두 도구 요청

두 도구는 차례로 실행됩니다. 결과를 모델에게 돌려줄 때까지 c1과 c2가 각각 어느 요청의 ID인지 따라가세요.

운영 시간과 대출 기간을 함께 물은 뒤 AIMessage의 tool_calls와 ToolMessage를 ID로 짝지으세요. 호출 순서와 ID는 실행마다 달라질 수 있습니다.

### 셀 06 · create_agent의 도구 왕복

각 도구가 실제로 불렸을 때의 반환값을 호출 ID와 함께 읽습니다.

셀 01의 같은 이름 함수는 사전을 반환해서 hours 키를 읽었습니다. 여기에서 새로 정의하는 도구는 문자열을 돌려줍니다. 메시지에서 호출 ID를 먼저 찾은 뒤 그 ID와 짝을 이룬 ToolMessage의 content를 읽으세요.


In [ ]:
def get_hours(day: str) -> str:
    """운영 시간을 찾는다. day는 weekday 또는 weekend다."""
    return hours[day]

def get_loan_rule() -> str:
    """가상 도서관 대출 일수를 찾는다."""
    return previous["record"]["text"].split("기간은 ", 1)[1].split("일", 1)[0]

agent = create_agent(model, tools=[get_hours, get_loan_rule])
with tracing_context(enabled=False):
    state = agent.invoke({"messages": [("user", "주말 시간과 대출 기간은?")]})
for message in state["messages"]:
    if message.type == "tool":
        print(message.tool_call_id, message.content)

call_VUKgnnwUI03xAwCk1gAWEyy1 10:00~16:00
call_bUVIHW5iC1D3tZ36EgWCj8Tp 14


54장에서 보관한 원문의 일수를 도구 반환값으로 확인하세요. 기본 자료이면 14이고 21일로 고친 원문이면 21입니다. 운영 시간과 대출 기간을 함께 물은 뒤 AIMessage의 tool_calls와 ToolMessage를 ID로 짝지으세요. 호출 순서와 ID는 실행마다 달라질 수 있습니다.

#### 도구 두 개를 등록하면 둘 다 실행될까요?

<mark>등록은 호출할 수 있는 함수 목록을 정하고 실제 순서는 모델 메시지의 tool_calls가 정합니다.</mark> 실제 모델이 요청한 함수 이름과 ID를 읽으세요. 두 도구를 한 메시지에서 요청할 수도 있습니다. 마지막 AI 메시지에는 새 요청이 없어 반복이 끝납니다. 결과 메시지 두 개와 마지막 답변에 쓰인 값을 짝지어 보세요.

선수 개념 · 54장 SQL 조회와 소유자 조건

#### create_agent의 두 번의 도구 왕복

```mermaid
flowchart TD
s["사용자 질문"] --> m["모델 호출"]
m --> d{"이번 AIMessage의 tool_calls?"}
d -->|시간 조회 요청| h["get_hours · 주말 시간"]
d -->|기간 조회 요청| l["get_loan_rule · 대출 일수"]
h --> t["ToolMessage · 호출 ID와 결과"]
l --> t
t --> m
d -->|없음| e["최종 AIMessage"]
```

운영 시간과 대출 기간을 함께 물은 뒤 AIMessage의 tool_calls와 ToolMessage를 ID로 짝지으세요. 호출 순서와 ID는 실행마다 달라질 수 있습니다.

## 복습과 실습 점검

### 과제 07 · 도구 결과에서 최종 답변에 쓰인 값 찾기

셀 06이 반환한 state에서 호출 ID가 c2인 도구 메시지를 찾아 도구 이름과 반환값을 출력하세요. 메시지의 위치는 실행 경로에 따라 달라질 수 있습니다.


In [ ]:
# state["messages"]에서 이름이 get_loan_rule인 도구 결과를 찾으세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>for message in state[&quot;messages&quot;]:
    if message.type == &quot;tool&quot; and message.name == &quot;get_loan_rule&quot;:
        print(message.name, message.content)</code></pre>
<p>실행 결과</p><pre><code>get_loan_rule 14</code></pre>
<p>c2가 가리키는 도구는 대출 규정을 조회했습니다. 마지막 AI 메시지의 문장을 바꾸더라도 이 ToolMessage에는 실행한 함수의 반환값이 남습니다.</p>
</details>


## create_agent가 검색 도구를 다시 부르는 순간

운영 시간과 대출 기간을 함께 물은 뒤 AIMessage의 tool_calls와 ToolMessage를 ID로 짝지으세요. 호출 순서와 ID는 실행마다 달라질 수 있습니다.

### 셀 09 · 도구 호출 전후의 노드 업데이트 읽기

주말 시간과 대출 기간을 물었던 agent를 같은 입력으로 실행합니다. 최종 상태 한 덩어리를 읽었던 invoke와 달리 stream의 updates에서는 방금 일한 노드 이름이 보입니다.

In [7]:
question_input = {"messages": [("user", "주말 시간과 대출 기간은?")]}
with tracing_context(enabled=False):
    for update in agent.stream(question_input, stream_mode="updates"):
        for node_name, change in update.items():
            message = change["messages"][-1]
            if message.type == "tool":
                print(node_name, message.tool_call_id, message.content)
            elif message.tool_calls:
                print(node_name, message.tool_calls[0]["name"])
            else:
                print(node_name, message.content)

model get_hours
tools c1 10:00~16:00
model get_loan_rule
tools c2 14
model 주말은 10:00~16:00이며 대출은 14일입니다.


처음에는 model이 get_hours를 요청하고 tools가 c1의 운영 시간을 돌려줍니다. 이어 model이 get_loan_rule을 요청하고 tools가 c2에 14를 붙입니다. 마지막 model의 메시지에는 새 호출 요청이 없어 실행이 끝납니다. 도구 이름은 실행 전 요청에도 보이므로 실제 실행 여부는 tools의 결과로 확인하세요.

### 과제 10 · 같은 도구를 등록하고도 검색 없이 끝내기

인사 한 건에는 도구 요청이 없는 AIMessage 한 건만 준비하세요. 기존 도구 두 개를 똑같이 등록한 새 에이전트를 만들고, 인사를 보낸 뒤 전체 메시지 타입과 마지막 문장을 출력하세요.

In [ ]:
greeting_input = {"messages": [("user", "안녕하세요!")]}
# 실제 모델과 기존 도구로 인사를 처리하고 도구 사용 여부를 확인하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>greeting_input = {&quot;messages&quot;: [(&quot;user&quot;, &quot;안녕하세요!&quot;)]}
greeting_greeting_agent = create_agent(model, tools=[get_hours, get_loan_rule])
with tracing_context(enabled=False):
    history = greeting_agent.invoke(greeting_input)[&quot;messages&quot;]
print([message.type for message in history])
print(history[-1].content)</code></pre>
<p>실행 결과</p><pre><code>[&#x27;human&#x27;, &#x27;ai&#x27;]
도서관 안내입니다.</code></pre>
<p>운영 시간과 대출 기간을 함께 물은 뒤 AIMessage의 tool_calls와 ToolMessage를 ID로 짝지으세요. 호출 순서와 ID는 실행마다 달라질 수 있습니다.</p>
</details>

### 셀 11 · 실제 모델의 선택을 확인하는 선택 셀

도구 호출을 지원하는 수업용 채팅 모델을 준비했다면 class_chat에 넣으세요. OpenAI 계정으로 실습할 때는 커널에 langchain-openai와 OPENAI_API_KEY를 설정하고 주석의 모델 이름을 수업에서 쓰는 값으로 바꾸세요. 기본 설정에서는 외부 요청을 보내지 않습니다.

In [ ]:
live_agent = create_agent(model, tools=[get_hours, get_loan_rule])
with tracing_context(enabled=False):
    run = live_agent.invoke(question_input)
for message in run["messages"]:
    if message.type == "tool":
        print(message.name, message.content)
print(run["messages"][-1].content)

get_hours 10:00~16:00
get_loan_rule 14
주말 운영 시간은 10:00부터 16:00까지이며, 대출 기간은 14일입니다.


실제 모델을 연결했다면 최종 문장에 운영 시간이 있는지와 get_hours의 결과가 기록에 있는지를 각각 확인하세요. 답에 시간이 있어도 도구를 거쳤다는 증거는 그 앞의 ToolMessage에서 찾아야 합니다.

<mark>도구를 등록한 횟수와 도구를 실행한 횟수는 다른 값입니다.</mark> 질문이 달라져도 등록 목록은 같습니다. 두 실습에서 달라진 것은 모델이 반환한 AIMessage의 tool_calls와 그 결과로 지나간 노드입니다.

56장에서는 모델이 요청한 호출 중 정책이 실제 함수 실행을 허용한 건수를 따로 확인합니다.

## @tool과 invoke로 SQL 원문을 도구 결과로 돌려주기

54장에서 고친 규정을 서비스의 조회 도구로 등록합니다. 먼저 도구를 직접 한 번 호출해 비교값을 만드세요. 본문의 create_agent 예에서는 모델이 제안한 인자와 도구 메시지를 따로 확인합니다.

In [ ]:
from langchain_core.tools import tool
record = previous["record"]
@tool
def lookup_loan(doc_id: str) -> str:
    """문서 번호로 대출 규정 원문을 찾는다."""
    return record["text"] if doc_id == record["id"] else ""
args = {"doc_id": record["id"]}
result = lookup_loan.invoke(args)
print(lookup_loan.name, result)

lookup_loan 대출 기간은 14일입니다.


invoke가 doc_id를 함수에 전달합니다. 54장에서 21일을 남겼다면 이 결과에도 21일이 보입니다. 이 한 줄은 직접 호출한 결과입니다.

### 직접 확인하기

등록되지 않은 문서 번호를 넣은 요청 기록을 만들어 보세요. 원문이 없을 때 빈 문자열과 정상 응답을 구분할 수 있어야 합니다.

In [ ]:
# 다른 번호로 invoke를 호출하고 repr로 빈 문자열을 표시하세요.

<details><summary>정답 코드와 실행 결과 보기</summary><pre><code>print(repr(lookup_loan.invoke({&quot;doc_id&quot;: &quot;missing&quot;})))</code></pre><p>기본 자료로 실행한 결과</p><pre><code>&#x27;&#x27;</code></pre><p>도구 이름은 같아도 요청 인자가 다르면 반환값이 달라집니다. 다음 장에서는 정상 요청의 인자와 결과를 사용합니다.</p></details>

### 56장에 넘길 결과 저장하기

56장은 같은 조회를 두 번 요청하는 상황에서 실행 한도를 적용합니다. 저장 셀을 다시 실행하면 같은 파일에 이번 결과를 기록합니다.

In [ ]:
product = {"record": record, "call": {"name": lookup_loan.name, "args": args, "result": result}}
target = work / "agent-55.json"
target.write_text(json.dumps(product, ensure_ascii=False, indent=2), encoding="utf-8")
print(target.name)

agent-55.json
